# Continuous age regression from gait

This is a parallel age-from-gait experiment. It does not feed age into the primary stroke classifier. The notebook first audits the wider age coverage in the complete local Voisard release, then compares an engineered-feature baseline with a GPU Inception-style regression model using healthy Voisard participants only. All splits are participant-level.

In [1]:
import json
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, Dataset

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED = PROJECT_ROOT / 'data' / 'processed'; INTERIM = PROJECT_ROOT / 'data' / 'interim'; sys.path.insert(0, str(PROJECT_ROOT / 'src'))
MAG_PATH = PROCESSED / 'validated_acceleration_magnitude_windows_float32.npy'
METADATA_PATH = PROCESSED / 'validated_window_metadata.csv'; MANIFEST_PATH = INTERIM / 'ml_readiness_manifest.csv'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); torch.set_num_threads(4)
EPOCHS = 8; BATCH_SIZE = 128
metadata = pd.read_csv(METADATA_PATH); manifest = pd.read_csv(MANIFEST_PATH); magnitude_windows = np.load(MAG_PATH, mmap_mode='r')
participant_age = manifest[manifest.dataset_id.eq('voisard_2025')].groupby('subject').age.first().astype(float).to_dict()
print('Device:', DEVICE)

Device: cuda


In [2]:
# Full local Voisard age coverage, including cohorts excluded from the primary binary task.
raw_root = PROJECT_ROOT / 'data' / 'raw' / 'voisard_2025' / 'data'
raw_rows = []
for path in raw_root.rglob('*_meta.json'):
    record = json.loads(path.read_text(encoding='utf-8'))
    relative = path.relative_to(raw_root).parts; age_value = record.get('age')
    raw_rows.append({'subject': record.get('subject'), 'age': float(age_value) if age_value is not None else np.nan, 'group': record.get('group'), 'cohort': relative[0] if relative else 'unknown'})
raw_trials = pd.DataFrame(raw_rows)
raw_participants = raw_trials.drop_duplicates('subject').copy()
raw_participants['valid_age'] = raw_participants.age.between(18, 100, inclusive='both')
raw_participants['age_group'] = pd.cut(raw_participants.age, bins=[17, 29, 39, 49, 59, 69, 79, 100], labels=['18-29', '30-39', '40-49', '50-59', '60-69', '70-79', '80+'])
coverage = raw_participants[raw_participants.valid_age].groupby(['cohort', 'group', 'age_group'], observed=True).size().rename('participants').reset_index()
coverage.to_csv(PROCESSED / 'age_coverage_audit.csv', index=False)
print('Full release valid-age range:', raw_participants.loc[raw_participants.valid_age, 'age'].min(), 'to', raw_participants.loc[raw_participants.valid_age, 'age'].max())
print(coverage.to_string(index=False))

Full release valid-age range: 18.0 to 90.0
 cohort   group age_group  participants
healthy healthy     18-29            29
healthy healthy     30-39            14
healthy healthy     40-49            10
healthy healthy     50-59             5
healthy healthy     60-69             7
healthy healthy     70-79             2
healthy healthy       80+             6
  neuro   neuro     18-29             3
  neuro   neuro     30-39             2
  neuro   neuro     40-49            11
  neuro   neuro     50-59            40
  neuro   neuro     60-69            51
  neuro   neuro     70-79            18
  neuro   neuro       80+            17
  ortho   ortho     18-29             4
  ortho   ortho     30-39             3
  ortho   ortho     40-49             5
  ortho   ortho     50-59             5
  ortho   ortho     60-69             9
  ortho   ortho     70-79             9
  ortho   ortho       80+             9


In [3]:
# Healthy-only primary age-regression table. One row per participant prevents trial count from acting as sample size.
FEATURES = ['cadence_steps_per_min', 'stride_time_mean_s', 'stride_time_cv_mean', 'lb_accel_rms', 'foot_accel_rms_mean', 'he_accel_rms']
from features import voisard
trial_features = voisard.build_feature_table()
participant_features = trial_features[trial_features.label.eq('HS')].groupby('subject', as_index=False)[FEATURES].mean()
participant_features['age'] = participant_features.subject.map(participant_age)
participant_features = participant_features.dropna(subset=['age'] + FEATURES).reset_index(drop=True)
participant_table = metadata[metadata.dataset_id.eq('voisard_2025') & metadata.label.eq('healthy')][['participant_key']].drop_duplicates()
participant_table['subject'] = participant_table.participant_key.str.split(':').str[-1]
participant_table = participant_table.merge(participant_features, on='subject', how='inner')
window_meta = metadata[metadata.participant_key.isin(participant_table.participant_key)].copy()
window_meta['age'] = window_meta.participant_key.map(participant_table.set_index('participant_key').age)
print('Healthy participants:', len(participant_table), '| windows:', len(window_meta), '| age range:', participant_table.age.min(), 'to', participant_table.age.max())

Healthy participants: 72 | windows: 1039 | age range: 18.0 to 87.0


In [4]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def pooled_stats(indices):
    total = np.zeros(3, dtype='float64'); total_sq = np.zeros(3, dtype='float64'); count = 0
    for start in range(0, len(indices), 512):
        batch = np.asarray(magnitude_windows[indices[start:start + 512]], dtype='float32')
        total += batch.sum(axis=(0, 1)); total_sq += np.square(batch).sum(axis=(0, 1)); count += batch.shape[0] * batch.shape[1]
    mean = total / count; std = np.sqrt(np.maximum(total_sq / count - mean ** 2, 1e-8)); return mean.astype('float32'), std.astype('float32')

class AgeDataset(Dataset):
    def __init__(self, indices, mean, std, target_mean, target_std, weights=None):
        self.indices = np.asarray(indices, dtype='int64'); self.mean = mean.reshape(1, 3); self.std = std.reshape(1, 3); self.target_mean = target_mean; self.target_std = target_std
        self.weights = np.ones(len(self.indices), dtype='float32') if weights is None else weights
    def __len__(self): return len(self.indices)
    def __getitem__(self, item):
        index = int(self.indices[item]); signal = ((np.asarray(magnitude_windows[index], dtype='float32') - self.mean) / self.std).T.copy()
        age = (float(window_meta.loc[index, 'age']) - self.target_mean) / self.target_std
        return torch.from_numpy(signal), torch.tensor(age, dtype=torch.float32), torch.tensor(float(self.weights[item])), torch.tensor(index)

class InceptionBlock(nn.Module):
    def __init__(self, in_channels, out_channels=16):
        super().__init__(); bottleneck = min(32, in_channels); self.bottleneck = nn.Conv1d(in_channels, bottleneck, 1, bias=False)
        self.branches = nn.ModuleList([nn.Conv1d(bottleneck, out_channels, 7, padding=3, bias=False), nn.Conv1d(bottleneck, out_channels, 15, padding=7, bias=False), nn.Conv1d(bottleneck, out_channels, 25, padding=12, bias=False)])
        self.pool_branch = nn.Conv1d(in_channels, out_channels, 1, bias=False); self.bn = nn.BatchNorm1d(out_channels * 4); self.residual = nn.Conv1d(in_channels, out_channels * 4, 1, bias=False) if in_channels != out_channels * 4 else nn.Identity()
    def forward(self, x):
        z = self.bottleneck(x); branches = [branch(z) for branch in self.branches]; branches.append(self.pool_branch(nn.functional.max_pool1d(x, 3, stride=1, padding=1)))
        return nn.functional.gelu(self.bn(torch.cat(branches, dim=1)) + self.residual(x))

class AgeRegCNN(nn.Module):
    def __init__(self):
        super().__init__(); self.features = nn.Sequential(InceptionBlock(3), nn.MaxPool1d(2), InceptionBlock(64), nn.AdaptiveAvgPool1d(1)); self.regressor = nn.Sequential(nn.Flatten(), nn.Dropout(0.30), nn.Linear(64, 1))
    def forward(self, x): return self.regressor(self.features(x)).squeeze(1)

In [5]:
def participant_metrics(y, pred):
    return {'mae_years': mean_absolute_error(y, pred), 'rmse_years': np.sqrt(mean_squared_error(y, pred)), 'r2': r2_score(y, pred), 'spearman_rho': spearmanr(y, pred).statistic}

def feature_baseline(train_pos, test_pos, model_name):
    train = participant_table.iloc[train_pos]; test = participant_table.iloc[test_pos]
    scaler = StandardScaler().fit(train[FEATURES]); x_train = scaler.transform(train[FEATURES]); x_test = scaler.transform(test[FEATURES])
    model = Ridge(alpha=10.0) if model_name == 'ridge_features' else RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=42, n_jobs=-1)
    model.fit(x_train, train.age); pred = model.predict(x_test)
    return participant_metrics(test.age.to_numpy(), pred)

def train_cnn(train_pos, test_pos, fold, seed):
    set_seed(seed); train_keys = set(participant_table.iloc[train_pos].participant_key); test_keys = set(participant_table.iloc[test_pos].participant_key)
    train_indices = window_meta.index[window_meta.participant_key.isin(train_keys)].to_numpy(); test_indices = window_meta.index[window_meta.participant_key.isin(test_keys)].to_numpy()
    mean, std = pooled_stats(train_indices); target_mean = float(participant_table.iloc[train_pos].age.mean()); target_std = float(participant_table.iloc[train_pos].age.std())
    counts = window_meta.loc[train_indices].groupby('participant_key').size(); weights = window_meta.loc[train_indices].participant_key.map(1.0 / counts).to_numpy(); weights = (weights / weights.mean()).astype('float32')
    train_loader = DataLoader(AgeDataset(train_indices, mean, std, target_mean, target_std, weights), batch_size=BATCH_SIZE, shuffle=True, num_workers=0); test_loader = DataLoader(AgeDataset(test_indices, mean, std, target_mean, target_std), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    model = AgeRegCNN().to(DEVICE); optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    for _ in range(EPOCHS):
        model.train()
        for signals, targets, weights_batch, _ in train_loader:
            optimizer.zero_grad(); pred = model(signals.to(DEVICE)); loss = (nn.functional.smooth_l1_loss(pred, targets.to(DEVICE), reduction='none') * weights_batch.to(DEVICE)).mean(); loss.backward(); optimizer.step()
    model.eval(); window_rows = []
    with torch.no_grad():
        for signals, _, _, indices in test_loader:
            pred = (model(signals.to(DEVICE)).cpu().numpy() * target_std) + target_mean; window_rows.extend(zip(indices.numpy(), pred))
    pred_map = {int(i): float(p) for i, p in window_rows}; frame = window_meta.loc[test_indices, ['participant_key', 'age']].copy(); frame['pred'] = [pred_map[int(i)] for i in test_indices]
    grouped = frame.groupby(['participant_key', 'age'], as_index=False).pred.mean(); metrics = participant_metrics(grouped.age.to_numpy(), grouped.pred.to_numpy()); metrics.update({'model': 'inception_regression', 'fold': fold, 'seed': seed, 'participants': len(grouped)}); return grouped.assign(model='inception_regression', fold=fold, seed=seed), metrics

In [6]:
all_rows = []; metric_rows = []
for seed in [42, 52, 62]:
    splitter = KFold(n_splits=5, shuffle=True, random_state=seed)
    for fold, (train_pos, test_pos) in enumerate(splitter.split(participant_table)):
        for model_name in ['ridge_features', 'random_forest_features']:
            row = feature_baseline(train_pos, test_pos, model_name); row.update({'model': model_name, 'fold': fold, 'seed': seed, 'participants': len(test_pos)}); metric_rows.append(row)
        pred_frame, row = train_cnn(train_pos, test_pos, fold, seed); all_rows.append(pred_frame); metric_rows.append(row)
        print('seed', seed, 'fold', fold, 'CNN', {k: round(v, 3) if isinstance(v, float) else v for k, v in row.items() if k in ['mae_years', 'rmse_years', 'r2', 'spearman_rho']})
metrics = pd.DataFrame(metric_rows); predictions = pd.concat(all_rows, ignore_index=True)
print(metrics.groupby('model')[['mae_years', 'rmse_years', 'r2', 'spearman_rho']].agg(['mean', 'std']).round(3).to_string())

seed 42 fold 0 CNN {'mae_years': 14.242, 'rmse_years': np.float64(19.897), 'r2': 0.296, 'spearman_rho': np.float64(0.705)}


seed 42 fold 1 CNN {'mae_years': 15.637, 'rmse_years': np.float64(18.631), 'r2': -0.068, 'spearman_rho': np.float64(0.345)}


seed 42 fold 2 CNN {'mae_years': 10.738, 'rmse_years': np.float64(12.412), 'r2': 0.373, 'spearman_rho': np.float64(0.76)}


seed 42 fold 3 CNN {'mae_years': 14.154, 'rmse_years': np.float64(17.203), 'r2': 0.426, 'spearman_rho': np.float64(0.77)}


seed 42 fold 4 CNN {'mae_years': 11.826, 'rmse_years': np.float64(15.133), 'r2': 0.201, 'spearman_rho': np.float64(0.623)}


...

seed 62 fold 0 CNN {'mae_years': 11.694, 'rmse_years': np.float64(14.365), 'r2': 0.316, 'spearman_rho': np.float64(0.606)}


seed 62 fold 1 CNN {'mae_years': 15.546, 'rmse_years': np.float64(18.541), 'r2': 0.396, 'spearman_rho': np.float64(0.756)}


seed 62 fold 2 CNN {'mae_years': 15.532, 'rmse_years': np.float64(18.99), 'r2': -0.104, 'spearman_rho': np.float64(0.04)}


seed 62 fold 3 CNN {'mae_years': 9.435, 'rmse_years': np.float64(10.504), 'r2': 0.096, 'spearman_rho': np.float64(0.665)}


seed 62 fold 4 CNN {'mae_years': 15.687, 'rmse_years': np.float64(19.102), 'r2': 0.35, 'spearman_rho': np.float64(0.802)}
                       mae_years        rmse_years            r2        spearman_rho       
                            mean    std       mean    std   mean    std         mean    std
model                                                                                      
inception_regression      13.417  2.139     16.478  2.961  0.243  0.188        0.580  0.213
random_forest_features    13.004  2.394     15.840  3.240  0.258  0.347        0.611  0.194
ridge_features            12.857  1.336     15.458  1.859  0.310  0.216        0.643  0.123


In [7]:
metrics.to_csv(PROCESSED / 'continuous_age_regression_metrics.csv', index=False); predictions.to_csv(PROCESSED / 'continuous_age_regression_predictions.csv', index=False)
print('Saved continuous age regression outputs.')

Saved continuous age regression outputs.


## Interpretation gate

This regression estimates age-related gait structure in healthy participants. It does not establish that age should be an input to the stroke classifier. The wide full-release age range includes other neurological and orthopedic cohorts, but those labels must not be pooled into the healthy age-regression target without a separate scientific question.